<a href="https://colab.research.google.com/github/marlazeee/Digital_Modulation_Simulator-/blob/main/Digital_Modulation_Sim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gradio --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gradio as gr

In [ ]:
# Generate a random binary bitstream of length n
def generate_bits(n):
    return np.random.randint(0, 2, n)  # Each bit is randomly 0 or 1

# Map bits using NRZ format: 0 → -1, 1 → +1
def nrz_map(bits):
    return 2 * bits - 1  # Multiply by 2 and subtract 1

# Upsample the symbol stream by repeating each symbol
def upsample(symbols, samples_per_symbol):
    return np.repeat(symbols, samples_per_symbol)  # Stretch each symbol across multiple samples

# Generate a cosine carrier signal
def generate_carrier(frequency, fs, duration):
    # Create time axis based on sampling frequency and total duration
    t = np.arange(0, duration, 1/fs)

    # Generate cosine wave at the given carrier frequency
    carrier = np.cos(2 * np.pi * frequency * t)

    # Return both the time axis and carrier waveform
    return t, carrier

def add_awgn(I, Q, snr_db):
    # Convert SNR from decibels to linear scale
    snr_linear = 10**(snr_db / 10)
    # Calculate the average power of the signal
    power_signal = np.mean(I**2 + Q**2)
    # Calculate the required noise power to achieve the desired SNR
    noise_power = power_signal / snr_linear
    # Calculate standard deviation of Gaussian noise (per dimension: I and Q)
    noise_std = np.sqrt(noise_power / 2)

    # Add Gaussian noise to I and Q components
    I_noisy = I + np.random.normal(0, noise_std, len(I))
    Q_noisy = Q + np.random.normal(0, noise_std, len(Q))
    return I_noisy, Q_noisy

In [ ]:
# BPSK Mapping Function
def bpsk_map(bits):
    # Map bits to in-phase (I) symbols using NRZ: 0 → -1, 1 → +1
    I = 2 * bits - 1
    # BPSK has no quadrature (Q) component, so Q = 0
    Q = np.zeros_like(I)
    # Return I and Q components (Q is all zeros for BPSK)
    return I, Q


# QPSK Mapping Function
def qpsk_map(bits):
    # Truncate bitstream to even length and group into pairs
    bit_pairs = bits[:(len(bits)//2)*2].reshape(-1, 2)
    # Initialize empty lists for in-phase (I) and quadrature (Q) components
    I, Q = [], []
    # Gray-coded QPSK constellation mapping (normalized later)
    mapping = {
        (0,0): (+1, +1),
        (0,1): (+1, -1),
        (1,1): (-1, -1),
        (1,0): (-1, +1),
    }

    # Loop through each bit pair and map to I/Q values
    for pair in bit_pairs:
        i, q = mapping[tuple(pair)]
        # Normalize to unit energy by dividing by √2
        I.append(i / np.sqrt(2))
        Q.append(q / np.sqrt(2))

    # Return I and Q components as numpy arrays
    return np.array(I), np.array(Q)

def psk_map(bits, M):
  k = int(np.log2(M))
  bits = bits[:len(bits)//k * k].reshape(-1, k)
  indices = np.array([int("".join(str(b) for b in group), 2) for group in bits])
  angles = 2 * np.pi * indices / M
  I = np.cos(angles)
  Q = np.sin(angles)
  return I, Q

def qam_map(bits, M):
  k = int(np.log2(M))
  bits = bits[:len(bits)//k * k].reshape(-1, k)
  m = int(np.sqrt(M))
  levels = np.array([-3, -1, +1, +3]) if m == 4 else np.linspace(-(m-1), m-1, m)
  def bits_to_int(b):
    return int("".join(str(x) for x in b), 2)
  I, Q = [], []
  for group in bits:
    i_idx = bits_to_int(group[:k//2])
    q_idx = bits_to_int(group[k//2:])
    I.append(levels[i_idx])
    Q.append(levels[q_idx])
  I, Q = np.array(I), np.array(Q)
  norm = np.sqrt(np.mean(I**2 + Q**2))
  return I / norm, Q / norm

In [ ]:
def bpsk_modulate(bits, fc=50, fs=1000, samples_per_symbol=50):
    # Map bits to symbols using NRZ mapping: 0 → -1, 1 → +1
    symbols = 2 * bits - 1

    # Upsample the symbol stream (stretch each symbol across many samples)
    symbols_up = np.repeat(symbols, samples_per_symbol)

    # Create a time axis matching the length of the upsampled signal
    t = np.arange(0, len(symbols_up)/fs, 1/fs)

    # Generate the carrier signal (cosine wave at carrier frequency fc)
    carrier = np.cos(2 * np.pi * fc * t)

    # Multiply symbols by carrier → BPSK modulation
    return t, symbols_up * carrier[:len(symbols_up)]


def qpsk_modulate(bits, fc=50, fs=1000, samples_per_symbol=50):
    # Ensure even number of bits and reshape into bit pairs (2 bits per QPSK symbol)
    bit_pairs = bits[:len(bits)//2*2].reshape(-1, 2)

    # Gray-coded QPSK mapping: assign I/Q values to each bit pair
    mapping = {
        (0,0): (+1, +1),
        (0,1): (+1, -1),
        (1,1): (-1, -1),
        (1,0): (-1, +1),
    }

    # Lists to store I and Q components
    I, Q = [], []

    # Apply mapping and normalize by √2 to maintain constant average power
    for pair in bit_pairs:
        i, q = mapping[tuple(pair)]
        I.append(i / np.sqrt(2))
        Q.append(q / np.sqrt(2))

    # Upsample the I and Q streams
    I_up = np.repeat(I, samples_per_symbol)
    Q_up = np.repeat(Q, samples_per_symbol)

    # Create a matching time axis
    t = np.arange(0, len(I_up)/fs, 1/fs)

    # Generate QPSK waveform:
    #   I component modulates cosine (in-phase)
    #   Q component modulates sine (quadrature), with a minus sign
    x = I_up * np.cos(2 * np.pi * fc * t) - Q_up * np.sin(2 * np.pi * fc * t)

    return t, x

# bits: input stream of 0s and 1s
# fc: carrier frequency
# fs: sampling frequency
# A0: amplitude bit for 0
# A1: amplitude bit for 1
def ask_modulate(bits, bitrate=1000, fc=100000, fs=200000, A0=0.2, A1=1.0):
  bits = np.array(bits)
  Tb = 1/bitrate
  time = Tb*len(bits)    # total time
  t = np.arrange(0, time, 1/fs)     # time vector

  # mapping each bit to amplitude
  amplitude = np.repeat(A1*(bits==1) + A0*(bits==0), int(fs*Tb))

  carrier = np.sin(2 * np.pi * fc * t)    # carrier signal

  # modulation
  ask_signal = amplitude * carrier

  return t, ask_signal


In [ ]:
def bpsk_ber_sim(snr_range_db, num_bits=10000):
    bers = []
    for snr_db in snr_range_db:
        bits = np.random.randint(0, 2, num_bits)
        symbols = 2 * bits - 1  # NRZ mapping
        snr_linear = 10**(snr_db / 10)
        noise_std = np.sqrt(1 / (2 * snr_linear))
        noise = np.random.normal(0, noise_std, num_bits)
        received = symbols + noise
        detected = (received > 0).astype(int)
        errors = np.sum(bits != detected)
        ber = errors / num_bits
        bers.append(ber)
    return bers


In [ ]:
# ASK Mapping Function
def ask_map(bits, A0=0.2, A1=1.0):
  #k = int(np.log2(M))  # bits per symbol
  #bits = bits[:len(bits)//k * k]
  #bit_groups = bits.reshape(-1, k)
  bits = np.array(bits)
  return A1*(bits==1) + A0*(bits==0)


In [ ]:
# 8-PSK Mapping Function
def psk_map(bits, M):
    k = int(np.log2(M))  # bits per symbol
    bits = bits[:len(bits)//k * k]  # truncate to multiple of k
    bit_groups = bits.reshape(-1, k)

    # Convert each group of k bits to an integer symbol index
    indices = np.array([int("".join(str(b) for b in group), 2) for group in bit_groups])

    # Map symbol index to angle on unit circle
    angles = 2 * np.pi * indices / M
    I = np.cos(angles)
    Q = np.sin(angles)
    return I, Q

In [ ]:
# QAM Mapping Function (16-QAM, 64-QAM)
def qam_map(bits, M):
    k = int(np.log2(M))  # bits per symbol
    bits = bits[:len(bits)//k * k]
    bit_groups = bits.reshape(-1, k)

    m = int(np.sqrt(M))  # grid dimension (e.g., 4x4 for 16-QAM)

    # Gray-code-like level assignment
    levels = np.array([-3, -1, +1, +3]) if m == 4 else np.linspace(-(m-1), m-1, m)

    def bits_to_int(b):
        return int("".join(str(x) for x in b), 2)

    I, Q = [], []
    for group in bit_groups:
        i_idx = bits_to_int(group[:k//2])
        q_idx = bits_to_int(group[k//2:])
        I.append(levels[i_idx])
        Q.append(levels[q_idx])

    # Normalize to unit power
    I = np.array(I)
    Q = np.array(Q)
    norm = np.sqrt(np.mean(I**2 + Q**2))
    return I / norm, Q / norm


In [ ]:
def bpsk_ber(snr_db, num_bits=10000):
    bits = np.random.randint(0, 2, num_bits)

    # BPSK Mapping: 0→-1, 1→+1
    symbols = 2 * bits - 1

    # Add noise
    snr_linear = 10**(snr_db / 10)
    noise_power = 1 / snr_linear
    noise = np.random.normal(0, np.sqrt(noise_power), num_bits)
    received = symbols + noise

    # Decision: threshold at 0
    detected_bits = (received > 0).astype(int)

    # Compare
    errors = np.sum(bits != detected_bits)
    ber = errors / num_bits
    return ber


In [ ]:
def visualize_constellation(mod_type, snr_db, num_bits):
    bits = np.random.randint(0, 2, num_bits)

    if mod_type == "BPSK":
        I, Q = 2*bits - 1, np.zeros_like(bits)
    elif mod_type == "QPSK":
        I, Q = qpsk_map(bits)
    elif mod_type == "8-PSK":
        I, Q = psk_map(bits, 8)
    elif mod_type == "16-QAM":
        I, Q = qam_map(bits, 16)
    elif mod_type == "64-QAM":
        I, Q = qam_map(bits, 64)
    else:
        return None

    I_n, Q_n = add_awgn(I, Q, snr_db)

    # Plot constellation
    fig, ax = plt.subplots(figsize=(5,5))
    ax.scatter(I_n, Q_n, alpha=0.5)
    ax.set_title(f"{mod_type} Constellation (SNR={snr_db} dB)")
    ax.axhline(0, color='gray')
    ax.axvline(0, color='gray')
    ax.set_xlabel("In-phase (I)")
    ax.set_ylabel("Quadrature (Q)")
    ax.grid(True)

    return fig


In [ ]:
mod_theory = {
    "ASK": "ASK (Amplitude Shift Keying) - changes the amplitude of the carrier signal",
    "BPSK": "BPSK (Binary Phase Shift Keying) - shifts the phase of the carrier wave by 180°. \nMaps bit 0 to phase 0° and bit 1 to phase 180°. Only 2 constellation points on real axis. Very robust.",
    "QPSK": "QPSK (Quadrature PSK) - changes phase of the carrier wave.\nMaps 2 bits into one of 4 phases (0°, 90°, 180°, 270°). Doubles data rate of BPSK using I/Q components.",
    "8-PSK": "8-PSK - changes the phase of a carier wave at 8 different angles, each unique phase is represented by 3 bits.\nUses 3 bits per symbol with 8 evenly spaced constellation points on the unit circle. Higher spectral efficiency but more noise-sensitive.",
    "16-QAM": "16-QAM (16-Quadrature Amplitude Modulation) - changes amplitude and phase of carrier wave to create 16 unique combinations.\nUses a 4x4 grid of I/Q points. Each symbol carries 4 bits. Higher capacity, moderate noise tolerance.",
    "64-QAM": "64-QAM (64-Quadrature Amplitude Modulation) - changes amplitude and phase of carrier wave to create 64 unique combinations.\nUses a dense 8x8 grid with 6 bits per symbol. Very high data rates, but very sensitive to noise."
}

In [ ]:
def generate_constellation_and_waveform(mod_type, snr_db, num_bits):
    bits = generate_bits(num_bits)

    if mod_type == "ASK":
        I, Q = ask_map(bits)
        t, waveform = ask_modulate(bits)
    elif mod_type == "BPSK":
        I, Q = bpsk_map(bits)
        t, waveform = bpsk_modulate(bits)
    elif mod_type == "QPSK":
        I, Q = qpsk_map(bits)
        t, waveform = qpsk_modulate(bits)
    elif mod_type == "8-PSK":
        I, Q = psk_map(bits, 8)
        t = np.arange(len(I))
        waveform = I
    elif mod_type == "16-QAM":
        I, Q = qam_map(bits, 16)
        t = np.arange(len(I))
        waveform = I
    elif mod_type == "64-QAM":
        I, Q = qam_map(bits, 64)
        t = np.arange(len(I))
        waveform = I
    else:
        return None, None, "Unsupported modulation."

    I_n, Q_n = add_awgn(I, Q, snr_db)

    # Constellation plot
    fig1, ax1 = plt.subplots(figsize=(5, 5))
    ax1.scatter(I_n, Q_n, alpha=0.5)
    ax1.set_title(f"{mod_type} Constellation (SNR={snr_db} dB)")
    ax1.axhline(0, color='gray')
    ax1.axvline(0, color='gray')
    ax1.set_xlabel("In-phase (I)")
    ax1.set_ylabel("Quadrature (Q)")
    ax1.grid(True)

    # Waveform plot
    fig2, ax2 = plt.subplots(figsize=(10, 3))
    ax2.plot(t[:500], waveform[:500])
    ax2.set_title(f"{mod_type} Waveform Preview")
    ax2.set_xlabel("Time")
    ax2.set_ylabel("Amplitude")
    ax2.grid(True)

    theory = mod_theory.get(mod_type, "No theory available for this modulation.")

    return fig1, fig2, theory


In [ ]:
# Visualization block
import gradio as gr

mod_select = gr.Dropdown(["BPSK", "QPSK", "8-PSK", "16-QAM", "64-QAM"], label="Modulation Type")
snr_slider = gr.Slider(0, 30, value=10, label="SNR (dB)")
bit_input = gr.Number(value=200, label="Number of Bits")

with gr.Blocks() as demo:
    gr.Markdown("## 🛰️ Digital Modulation Visualizer")

    with gr.Row():
        with gr.Column(scale=1):
            mod_select = gr.Dropdown(["BPSK", "QPSK", "8-PSK", "16-QAM", "64-QAM"], label="Modulation Type")
            snr_slider = gr.Slider(0, 30, value=10, label="SNR (dB)")
            bit_input = gr.Number(value=200, label="Number of Bits")
            run_button = gr.Button("Visualize")

        with gr.Column(scale=3):
            with gr.Tabs():
                with gr.TabItem("Constellation"):
                    constellation_plot = gr.Plot()
                with gr.TabItem("Waveform"):
                    waveform_plot = gr.Plot()
                with gr.TabItem("Theory"):
                    theory_output = gr.Markdown()

    run_button.click(
        generate_constellation_and_waveform,
        inputs=[mod_select, snr_slider, bit_input],
        outputs=[constellation_plot, waveform_plot, theory_output]
    )

if __name__ == "__main__":
    demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bac60c599fa372245b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
